# NYC Taxi Fare Prediction

## Data Cleaning

This notebook focuses on cleaning the raw taxi trip dataset.

The objective is to:
- Identify invalid observations
- Apply reasonable cleaning rules
- Generate a clean dataset for feature engineering and modeling

The original dataset contains more than 55 million taxi trips.
A sample is used during development before applying the pipeline to the full dataset.

## Load Development Sample

The required libraries are imported before loading the same 100,000-row sample used during data understanding.

In [23]:
import pandas as pd
import numpy as np

In [24]:
DATA_PATH = "../data/raw/train.csv"

df = pd.read_csv(
    DATA_PATH,
    nrows=100_000,
    parse_dates=["pickup_datetime"]
)

df.head()


,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2009-06-15 17:26:21.0000001,4.5,2009-06-15 17:26:21+00:00,-73.844311,40.721319,-73.841610,40.712278,1
1,2010-01-05 16:52:16.0000002,16.9,2010-01-05 16:52:16+00:00,-74.016048,40.711303,-73.979268,40.782004,1
2,2011-08-18 00:35:00.00000049,5.7,2011-08-18 00:35:00+00:00,-73.982738,40.761270,-73.991242,40.750562,2
3,2012-04-21 04:30:42.0000001,7.7,2012-04-21 04:30:42+00:00,-73.987130,40.733143,-73.991567,40.758092,1
4,2010-03-09 07:51:00.000000135,5.3,2010-03-09 07:51:00+00:00,-73.968095,40.768008,-73.956655,40.783762,1


## Initial Dataset Check

The sample size and numerical summary are reviewed first to establish a baseline before any observations are removed.

In [25]:
initial_rows = len(df)

initial_rows

100000

In [26]:
df.describe()

,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
count,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000,100000.000000
mean,11.354652,-72.494682,39.914481,-72.490967,39.919053,1.673820
std,9.716777,10.693934,6.225686,10.471386,6.213427,1.300171
min,-44.900000,-736.550000,-74.007670,-84.654241,-74.006377,0.000000
25%,6.000000,-73.992041,40.734996,-73.991215,40.734182,1.000000
50%,8.500000,-73.981789,40.752765,-73.980000,40.753243,1.000000
75%,12.500000,-73.966982,40.767258,-73.963433,40.768166,2.000000
max,200.000000,40.787575,401.083332,40.851027,404.616667,6.000000


### Initial Findings

Before cleaning:

- Number of records: 100,000
- Number of features: 8

Potential issues identified:
- Negative fare values
- Invalid geographic coordinates
- Zero passenger count

### Missing Value Check

Missing values are checked before applying row-level cleaning rules.

In [27]:
df.isnull().sum()

key                  0
fare_amount          0
pickup_datetime      0
pickup_longitude     0
pickup_latitude      0
dropoff_longitude    0
dropoff_latitude     0
passenger_count      0
dtype: int64

No missing values were found in the development sample.

No missing value treatment is required at this stage.

## Fare Amount Cleaning

Invalid fare records are inspected before the cleaning rule is applied.

In [28]:
negative_fares = df[df["fare_amount"] <= 0]

negative_fares.shape

(12, 8)

In [29]:
negative_fares.head(12)

,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
2039,2010-03-09 23:37:10.0000005,-2.9,2010-03-09 23:37:10+00:00,-73.789450,40.643498,-73.788665,40.641952,1
2486,2015-03-22 05:14:27.0000001,-2.5,2015-03-22 05:14:27+00:00,-74.000031,40.720631,-73.999809,40.720539,1
10002,2010-02-15 14:26:01.0000003,0.0,2010-02-15 14:26:01+00:00,-73.987115,40.738808,-74.005911,40.713960,1
13032,2013-08-30 08:57:10.0000002,-3.0,2013-08-30 08:57:10+00:00,-73.995062,40.740755,-73.995885,40.741357,4
27891,2015-05-15 21:40:28.00000010,0.0,2015-05-15 21:40:28+00:00,-74.077927,40.805714,-74.077919,40.805721,1
28839,2013-08-11 13:39:10.0000001,-2.5,2013-08-11 13:39:10+00:00,-73.785260,40.648442,0.000000,0.000000,1
36722,2015-04-30 15:19:45.0000003,-2.5,2015-04-30 15:19:45+00:00,-73.952187,40.790112,-73.950043,40.792839,1
42337,2015-03-09 10:29:46.0000004,-5.0,2015-03-09 10:29:46+00:00,-73.990974,40.755985,-73.980820,40.759869,1
47302,2010-03-18 19:13:39.0000002,0.0,2010-03-18 19:13:39+00:00,-73.942346,40.806269,-73.942463,40.806129,1
56748,2015-06-26 01:13:18.0000002,-5.0,2015-06-26 01:13:18+00:00,-73.979797,40.743240,-73.981216,40.737240,6


### Fare Cleaning Rule

Taxi fare should represent a positive monetary value.

Therefore:

- Records with `fare_amount <= 0` are considered invalid.
- These observations will be removed.

In [30]:
df = df[df["fare_amount"] > 0]

len(df)

99988

In [31]:
fare_removed = initial_rows - len(df)

fare_removed

12

## Geographic Coordinate Cleaning

Coordinate cleaning is performed in two stages: first removing geographically impossible values, then restricting trips to the NYC area.

### Valid Coordinate Ranges

In [32]:
df[
    [
        "pickup_longitude",
        "pickup_latitude",
        "dropoff_longitude",
        "dropoff_latitude"
    ]
].describe()

,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude
count,99988.000000,99988.000000,99988.000000,99988.000000
mean,-72.494509,39.914382,-72.491533,39.919360
std,10.694564,6.226052,10.469494,6.212511
min,-736.550000,-74.007670,-84.654241,-74.006377
25%,-73.992041,40.734996,-73.991215,40.734182
50%,-73.981789,40.752765,-73.980000,40.753243
75%,-73.966986,40.767258,-73.963438,40.768164
max,40.787575,401.083332,40.851027,404.616667


In [33]:
valid_coordinates = (
    df["pickup_longitude"].between(-180,180)
    &
    df["dropoff_longitude"].between(-180,180)
    &
    df["pickup_latitude"].between(-90,90)
    &
    df["dropoff_latitude"].between(-90,90)
)

In [34]:
(~valid_coordinates).sum()

np.int64(3)

In [35]:
df = df[valid_coordinates]

### NYC Area Filter

Valid global coordinates may still lie far outside New York City. Both pickup and dropoff points are therefore required to fall within a reasonable NYC bounding box.

In [36]:
NYC_LAT_MIN = 40.5
NYC_LAT_MAX = 41.0

NYC_LON_MIN = -74.3
NYC_LON_MAX = -73.7

In [37]:
nyc_area = (
    df["pickup_latitude"].between(
        NYC_LAT_MIN,
        NYC_LAT_MAX
    )
    &
    df["pickup_longitude"].between(
        NYC_LON_MIN,
        NYC_LON_MAX
    )
    &
    df["dropoff_latitude"].between(
        NYC_LAT_MIN,
        NYC_LAT_MAX
    )
    &
    df["dropoff_longitude"].between(
        NYC_LON_MIN,
        NYC_LON_MAX
    )
)

In [38]:
(~nyc_area).sum()

np.int64(2231)

In [39]:
df = df[nyc_area]

## Passenger Count Cleaning

Passenger count represents the number of passengers in each taxi trip.

Records with `passenger_count = 0` are considered invalid because a taxi fare prediction model should represent completed passenger trips.

A total of 358 records were identified and removed.

In [40]:
df["passenger_count"].value_counts().sort_index()

passenger_count
0      358
1    68011
2    14263
3     4210
4     2039
5     6870
6     2003
Name: count, dtype: int64

In [41]:
zero_passenger = df["passenger_count"] == 0

print("Zero passenger records:", zero_passenger.sum())

df = df[~zero_passenger]

Zero passenger records: 358


After removing zero-passenger records, the remaining observations represent trips with at least one recorded passenger.

## Cleaning Impact

The final row count is compared with the original sample to quantify the combined effect of all cleaning rules.

In [42]:
final_rows = len(df)

print("Before cleaning:", initial_rows)
print("After cleaning:", final_rows)
print("Removed:", initial_rows-final_rows)

Before cleaning: 100000
After cleaning: 97396
Removed: 2604


## Cleaning Summary

After applying cleaning rules:

- Invalid fares were removed.
- Impossible coordinates were removed.
- Trips outside the NYC area were filtered.
- Invalid passenger counts were removed.

The cleaned dataset is now ready for feature engineering.